In [ ]:
import geopandas as gpd
import pandas as pd

import cafo_iowa.db.session as s
import cafo_iowa.db.models as m

import geopandas as gpd
from sqlalchemy.orm import Session
from geoalchemy2.shape import to_shape
from shapely.geometry import shape

In [ ]:
# Function to fetch processed annotations along with parcel and permit IDs as GeoDataFrame
def get_processed_annotations_with_ids_as_gdf(session: Session):
    # Query to get processed annotations with related parcel and permit IDs
    results = (
        session.query(
            m.CFAnnotations.id.label("annotation_id"),
            m.Parcels.id.label("parcel_id"),
            m.Permits.id.label("permit_id"),
            m.CFAnnotations.clipped_annotation,
            m.CFAnnotations.clipped_annotation_empty,
            m.CFAnnotations.type,
            m.CFAnnotations.label,
            m.CFAnnotations.geometry,
            m.CFAnnotations.naip_qt_id,
        )
        .join(m.CFAnnotations.parcels)
        .join(m.Parcels.permits)
        .filter(m.Permits.id.isnot(None))
        .filter(m.CFAnnotations.type == "segment")
        .filter(
            m.CFAnnotations.clipped_annotation_empty.isnot(True)
        )  # Exclude empty annotations
        .all()
    )

    # Convert results into a GeoPandas GeoDataFrame
    data = []
    for result in results:
        # Extract relevant fields and convert geometry to shapely
        geom = to_shape(result.geometry) if result.geometry else None
        data.append(
            {
                "annotation_id": result.annotation_id,
                "parcel_id": result.parcel_id,
                "permit_id": result.permit_id,
                "clipped_annotation": result.clipped_annotation,
                "type": result.type,
                "label": result.label,
                "geometry": geom,
            }
        )

    # Create GeoDataFrame
    gdf = gpd.GeoDataFrame(data, geometry="geometry", crs="EPSG:26915")
    return gdf


# Example usage
session = s.get_session()

# Assuming `session` is your SQLAlchemy session
annotations_gdf = get_processed_annotations_with_ids_as_gdf(session)

In [ ]:
# get area in sqm
crs_orig = annotations_gdf.crs
annotations_gdf.crs = annotations_gdf.estimate_utm_crs()
annotations_gdf["area_sqm"] = annotations_gdf["geometry"].area
annotations_gdf.crs = crs_orig